In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import *
import json
import functools
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.get("job_parameters")

In [0]:
parameters = json.loads(dbutils.widgets.get("job_parameters"))
source_view_name = parameters.get("catalog") + "." + parameters.get("source_view")
target_table_name = parameters.get("catalog") + "." + parameters.get("target_table")
primary_keys = parameters.get("primary_keys")
deduplication_cols = parameters.get("deduplication_cols")
job_id = parameters.get("job_id")
parent_job = parameters.get("parent_job_id")
partition = parameters.get("partition")
default_value_flag = parameters.get("default_value_flag", True)

In [0]:
df = spark.read.table(source_view_name).filter(col("partition") == lit(partition))

In [0]:
cleansed_df = df.withColumns({
        "PK_null": functools.reduce(lambda a, b: a | b, (col(x).isNull() for x in primary_keys)),
        "dupe": row_number().over(Window.partitionBy(*primary_keys).orderBy(*deduplication_cols)) != 1
    })
enriched_df = cleansed_df.withColumns({
        "creation_timestamp": lit(current_timestamp()),
        "updation_timestamp": lit(current_timestamp()),
        "bad_record_flag": (col("PK_null") | col("dupe")).cast("string"),
        "bad_record_reason": (
            when(col("PK_null"), lit("MISSING PRIMARY KEY"))
            .when(col("dupe"), lit("DUPE"))
            .otherwise(lit(""))
        ),
        "job_run": lit(f"{job_id}{partition}"),
        "parent_job_run": lit(f"{parent_job}{partition}")
    })
source_df = enriched_df.select(*[col(x) for x in enriched_df.columns if x not in ["PK_null", "dupe"]])

In [0]:
if default_value_flag:
    print("Default Value Mapping: ", default_value_flag)
    default_value_mapping = {
        "int": 0,
        "double": 0.0,
        "string": "N/A"
    }
    source_df = source_df.select(*[coalesce(col(clmn), lit(default_value_mapping.get(dtype, None))).alias(clmn) for clmn, dtype in source_df.dtypes])

In [0]:
source_df.write \
    .format("delta") \
    .mode("append") \
    .insertInto(target_table_name)

In [0]:
operation_metrics = DeltaTable.forName(spark, target_table_name).history(1).select("operationMetrics.numOutputRows", "operationMetrics.numFiles", "operationMetrics.numOutputBytes")
display(operation_metrics)